# Анализ данных криптовалютного рынка для обнаружения волатильных паттернов и корреляций

**Щеглов Александр · ИД24-2**

## 1. Объект исследования и задачи начального анализа

**Объект исследования** — криптовалютный рынок, представленный выбранными монетами
и периодами наблюдения в датасетах.

Для описания его состояния используются цена, доходность, объём торгов и готовые
показатели волатильности. В качестве предполагаемых факторов влияния рассматриваются
настроения участников рынка, макроэкономические показатели и информационный фон.
Их фактическое влияние на этом этапе ещё не проверяется.

Основная единица наблюдения в рыночном CSV — **одна криптовалюта за один день**.
В остальных файлах это четырёхчасовая свеча BTC, новостная публикация или запись о торговом инструменте.

In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if isinstance(obj, (pd.DataFrame, pd.Series)) else obj)

DATA_DIR = Path(".")  # папка с четырьмя исходными файлами


## 2. Загрузка и назначение датасетов

### 2.1. Табличные рыночные данные — CSV

`crypto_market_cycles_2017_present.csv` содержит цены, объёмы, готовые показатели
доходности и волатильности, сведения о рыночных циклах и внешнем контексте.
Это основной источник для описания состояния рынка.

Для анализа читается **весь файл**. Первые пять строк используются только для просмотра.
CSV представляет табличные данные; сам файл не является реляционной базой данных.


In [2]:
path = DATA_DIR / "crypto_market_cycles_2017_present.csv"
df_market = pd.read_csv(path) if path.exists() else None
print("Рыночный CSV загружен." if df_market is not None else f"Нет файла: {path.name}")


Рыночный CSV загружен.


### 2.2. Иерархические данные — XML

`crypto-tradeing bots.xml` описывает торговые боты, стратегии и поддерживаемые блокчейны.
Записи внутри разделов преобразуются в таблицу для проверки заполненности.
Тип раздела сохраняется в `record_type`, идентификатор записи — в `record_id`.

На начальном этапе XML служит справочником инструментов. Без журнала сделок нельзя
установить, как активность конкретного бота связана с движением цены.


In [3]:
path = DATA_DIR / "crypto-tradeing bots.xml"
df_bots = None
if path.exists():
    root = ET.parse(path).getroot()
    rows = []
    for section in root.findall("section"):
        for record in section.findall("record"):
            row = {child.tag: " ".join(child.itertext()).strip() for child in record}
            row["record_id"] = record.get("id")
            row["record_type"] = row.get("record_type") or section.get("record_type")
            rows.append(row)
    df_bots = pd.DataFrame(rows)
    print("Корневой тег:", root.tag, "| Записей:", len(df_bots))
else:
    print(f"Нет файла: {path.name}")


Корневой тег: dataset | Записей: 220


### 2.3. Четырёхчасовые свечи BTC — JSON

`btcusdt_4h_fear_greed.json` содержит цены BTC и индекс страха и жадности.
JSON — полуструктурированный формат. Его записи преобразуются в таблицу.
В дальнейшем этот источник позволит рассмотреть рынок подробнее, чем дневной CSV.
Сейчас источники изучаются отдельно.


In [4]:
path = DATA_DIR / "btcusdt_4h_fear_greed.json"
df_json = None
if path.exists():
    with path.open(encoding="utf-8-sig") as f:
        records = json.load(f)
    if isinstance(records, dict):
        records = records.get("data", records.get("records", [records]))
    df_json = pd.DataFrame(records)
    print("JSON загружен. Записей:", len(df_json))
else:
    print(f"Нет файла: {path.name}")


JSON загружен. Записей: 14280


### 2.4. Новости — CSV с неструктурированным текстом

`crypto_news.csv` содержит заголовки, описания, источники и даты публикаций.
CSV задаёт табличную структуру, а поля `title` и `description` содержат
неструктурированный текст. Они представляют информационный фон рынка.
На этом этапе изучаются полнота текстов и состав источников.


In [5]:
path = DATA_DIR / "crypto_news.csv"
df_news = pd.read_csv(path) if path.exists() else None
print("Новостной CSV загружен." if df_news is not None else f"Нет файла: {path.name}")

datasets = {
    "Рынок (CSV)": df_market,
    "Боты (XML)": df_bots,
    "Свечи BTC (JSON)": df_json,
    "Новости (CSV)": df_news,
}
# Пустые строки и строки из пробелов тоже считаем пропусками.
for frame in datasets.values():
    if frame is not None:
        frame.replace(r"^\s*$", pd.NA, regex=True, inplace=True)


Новостной CSV загружен.


## 3. Первичное знакомство со структурой

Для каждого доступного файла посмотрим число строк и столбцов, названия полей,
первые записи и количество полных дубликатов.
Повторные записи пока только подсчитываются: перед удалением нужно понять, почему они появились.


In [6]:
overview_rows = []
for name, frame in datasets.items():
    overview_rows.append({
        "Датасет": name,
        "Файл подключён": frame is not None,
        "Строк": len(frame) if frame is not None else pd.NA,
        "Столбцов": len(frame.columns) if frame is not None else pd.NA,
    })
    if frame is not None:
        print(f"\n{name}")
        print("Столбцы:", list(frame.columns))
        print("Полных дубликатов сверх первой записи:", frame.astype("string").duplicated().sum())
        display(frame.head())
display(pd.DataFrame(overview_rows))



Рынок (CSV)
Столбцы: ['date', 'coin', 'open', 'high', 'low', 'close', 'volume', 'daily_return_pct', 'log_return', 'rolling_7d_vol', 'rolling_30d_vol', 'ath_usd', 'drawdown_from_ath_pct', 'sma_50d', 'sma_200d', 'golden_cross', 'fear_greed_value', 'fear_greed_label', 'fear_greed_zone', 'btc_dominance_pct', 'eth_dominance_pct', 'total_market_cap_usd', 'btc_market_cap_usd', 'eth_market_cap_usd', 'halving_epoch', 'days_since_halving', 'days_to_next_halving', 'is_halving_day', 'cycle_progress_pct', 'block_reward_btc', 'dxy_close', 'us10y_yield']
Полных дубликатов сверх первой записи: 0


,date,coin,open,high,low,close,volume,daily_return_pct,log_return,rolling_7d_vol,...,btc_market_cap_usd,eth_market_cap_usd,halving_epoch,days_since_halving,days_to_next_halving,is_halving_day,cycle_progress_pct,block_reward_btc,dxy_close,us10y_yield
0,2017-11-09,ADA,0.025160,0.035060,0.025006,0.032053,18716200,NaN,NaN,NaN,...,1.698874e+12,3.281062e+11,2,488,914,0,33.47,12.5,94.389999,2.400
1,2017-11-10,ADA,0.032219,0.033348,0.026451,0.027119,6766780,-15.393260,-0.167156,NaN,...,1.698874e+12,3.281062e+11,2,489,913,0,33.54,12.5,94.389999,2.400
2,2017-11-11,ADA,0.026891,0.029659,0.025684,0.027437,5532220,1.172610,0.011658,11.713839,...,1.698874e+12,3.281062e+11,2,490,912,0,33.61,12.5,94.389999,2.400
3,2017-11-12,ADA,0.027480,0.027952,0.022591,0.023977,7280250,-12.610706,-0.134797,8.870836,...,1.698874e+12,3.281062e+11,2,491,911,0,33.68,12.5,93.660004,2.354
4,2017-11-13,ADA,0.024364,0.026300,0.023495,0.025808,4419440,7.636480,0.073589,11.008518,...,1.698874e+12,3.281062e+11,2,492,910,0,33.74,12.5,93.660004,2.354



Боты (XML)
Столбцы: ['record_type', 'question', 'answer', 'bot_category', 'chain', 'section', 'model', 'provider', 'source_url', 'bot_url', 'tutorial_url', 'reliability', 'consensus', 'critical_note', 'record_id']
Полных дубликатов сверх первой записи: 0


,record_type,question,answer,bot_category,chain,section,model,provider,source_url,bot_url,tutorial_url,reliability,consensus,critical_note,record_id
0,bot_profile,"What is ArbitrageScanner, which chains does it...",Category: Arbitrage Scanner / CEX + DEX | Chai...,Arbitrage Scanner / CEX + DEX,40+ blockchains,bot_profile,<NA>,<NA>,https://telegramtrading.net/best-crypto-arbitr...,https://telegramtrading.net/ArbitrageScanner,<NA>,VERIFIED,False,False,profile_arbitragescanner
1,bot_profile,"What is Axiom, which chains does it support, a...",Category: DEX Terminal / Solana | Chains (2): ...,DEX Terminal / Solana,Solana | Hyperliquid (L1 perpetuals),bot_profile,<NA>,<NA>,https://telegramtrading.net/,<NA>,<NA>,VERIFIED,False,False,profile_axiom
2,bot_profile,"What is Banana Gun, which chains does it suppo...",Category: DEX Sniper / Multi-chain | Chain foc...,DEX Sniper / Multi-chain,Ethereum | Base | BNB Chain | MegaETH | Robinh...,bot_profile,<NA>,<NA>,https://telegramtrading.net/banana-gun-bot-rev...,https://telegramtrading.net/bananagun,<NA>,VERIFIED,False,False,profile_banana_gun
3,bot_profile,"What is Bloom, which chains does it support, a...",Category: DEX Sniper / Multi-chain | Chain foc...,DEX Sniper / Multi-chain,Solana | Ethereum | BNB Chain | Base | Hyperli...,bot_profile,<NA>,<NA>,https://telegramtrading.net/bloom-bot-review/,<NA>,<NA>,VERIFIED,False,False,profile_bloom
4,bot_profile,"What is BonkBot + Telemetry, which chains does...",Category: DEX Sniper / Solana | Chains (1): So...,DEX Sniper / Solana,Solana,bot_profile,<NA>,<NA>,https://telegramtrading.net/bonkbot-telemetry-...,https://telegramtrading.net/bonkbot,<NA>,VERIFIED,False,False,profile_bonkbot



Свечи BTC (JSON)
Столбцы: ['timestamp', 'open', 'close', 'high', 'low', 'Fear & Greed Index', 'Fear & Greed Classification']
Полных дубликатов сверх первой записи: 0


,timestamp,open,close,high,low,Fear & Greed Index,Fear & Greed Classification
0,2026-09-30 04:00:00,83640.0,83269.0,83685.0,83164.0,71.0,Greed
1,2026-09-30 00:00:00,83566.0,83640.0,83811.0,83368.0,71.0,Greed
2,2026-09-29 20:00:00,83080.0,83566.0,83663.0,82911.0,73.0,Greed
3,2026-09-29 16:00:00,84331.0,83080.0,84486.0,82936.0,73.0,Greed
4,2026-09-29 12:00:00,83978.0,84339.0,84442.0,83747.0,73.0,Greed



Новости (CSV)
Столбцы: ['url', 'title', 'description', 'thumbnail', 'createdAt', 'source']
Полных дубликатов сверх первой записи: 0


,url,title,description,thumbnail,createdAt,source
0,https://www.coindesk.com/markets/2026/08/15/ro...,Robot maker Unitree is going public. Hyperliqu...,Hyperliquid traders value Unitree at nearly $3...,https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 16:00:00 +0000",coindesk
1,https://www.coindesk.com/business/2026/08/15/s...,Swiss mega-bank UBS ramps up its Bitcoin expos...,"Direct holdings of IBIT also rose 12% to 407,8...",https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 15:56:05 +0000",coindesk
2,https://www.coindesk.com/policy/2026/08/15/why...,Why the world’s second-largest Bitcoin mining ...,The Energy Ministry enacted the year-round res...,https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 15:21:15 +0000",coindesk
3,https://www.coindesk.com/business/2026/08/15/p...,Paul Tudor Jones’ investment firm increases st...,"Calls fell 85.2% to 148,000 underlying shares,...",https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 15:12:29 +0000",coindesk
4,https://www.coindesk.com/business/2026/08/15/t...,The $11.2 billion in 2026 funding that killed ...,Dubai-based crypto lawyer Irina Heaver and her...,https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 14:00:00 +0000",coindesk


,Датасет,Файл подключён,Строк,Столбцов
0,Рынок (CSV),True,30120,32
1,Боты (XML),True,220,15
2,Свечи BTC (JSON),True,14280,7
3,Новости (CSV),True,413,6


## 4. Выбор числовых и категориальных характеристик

**Числовые характеристики** — измеряемые величины: цена, доходность, объём, значение индекса.
**Категориальные характеристики** — группы и метки: монета, источник новости, тип стратегии.

Тип хранения и смысл признака могут различаться. Например, `halving_epoch` хранится
числом, но обозначает категорию цикла; `golden_cross` и `is_halving_day` — бинарные признаки.
Даты относятся к временным характеристикам, заголовки и описания — к текстовым.
Идентификаторы и ссылки помогают различать записи, но сами по себе не описывают состояние рынка.

### 4.1. Характеристики основного рыночного CSV

| Признак | Тип по смыслу | Роль в аналитическом портрете |
|---|---|---|
| `close` | Числовой | Уровень цены закрытия |
| `daily_return_pct` | Числовой | Направление и величина дневного изменения цены |
| `rolling_7d_vol`, `rolling_30d_vol` | Числовые | Готовые оценки изменчивости на разных окнах |
| `drawdown_from_ath_pct` | Числовой | Снижение относительно максимума, указанного источником |
| `volume` | Числовой | Торговая активность; предполагаемый фактор связи с колебаниями |
| `fear_greed_value` | Числовой | Настроения участников рынка |
| `btc_dominance_pct` | Числовой | Доля Bitcoin и структура рынка |
| `total_market_cap_usd` | Числовой | Общий масштаб рынка |
| `dxy_close`, `us10y_yield` | Числовые | Внешний макроэкономический контекст |
| `days_since_halving` | Числовой | Положение наблюдения относительно халвинга Bitcoin |
| `coin` | Категориальный | Какая криптовалюта наблюдается |
| `fear_greed_label` | Категориальный | Группа рыночных настроений |
| `halving_epoch` | Категориальный | Эпоха халвинга Bitcoin |
| `golden_cross` | Бинарный категориальный | Технический сигнал или состояние тренда — уточнить по описанию источника |
| `is_halving_day` | Бинарный категориальный | Признак дня халвинга |

Таким образом, **цену, доходность, волатильность и просадку** выбираем для описания
состояния объекта. **Объём, настроения, рыночные и макроэкономические показатели**
рассматриваем как кандидатов для дальнейшего исследования связей.
Выбор признака ещё не доказывает его влияние.

Для первого этапа достаточно этого набора. `open`, `high`, `low` и остальные поля
сохраняются в исходной таблице и могут быть использованы позднее.
Перед сравнением готовых показателей нужно уточнить единицы объёма и методику расчёта волатильности.


### 4.2. Характеристики остальных источников

| Датасет | Числовые | Категориальные | Временные / текстовые и назначение |
|---|---|---|---|
| JSON BTC | `open`, `high`, `low`, `close`, `Fear & Greed Index` | `Fear & Greed Classification` | `timestamp` — время свечи; цены описывают состояние BTC, индекс — настроения |
| XML ботов | Готовые рыночные числовые признаки не выбираются | `record_type`, `bot_category`, `chain` | `record_id` — идентификатор; `question`, `answer` — текстовый контекст |
| Новости | На этом этапе числовые признаки не создаются | `source` | `createdAt` — дата; `title`, `description` — информационный фон; `url` — ссылка |

Не каждый датасет должен содержать оба типа признаков. XML полезен как справочник,
а новости — как текстовый источник. Их не нужно искусственно превращать в набор чисел
на первом этапе работы.


In [7]:
# Выбранные признаки. Берём только те, которые действительно есть в файлах.
numeric_features = {
    "Рынок (CSV)": ["close", "daily_return_pct", "rolling_7d_vol", "rolling_30d_vol",
                    "drawdown_from_ath_pct", "volume", "fear_greed_value",
                    "btc_dominance_pct", "total_market_cap_usd", "dxy_close",
                    "us10y_yield", "days_since_halving"],
    "Свечи BTC (JSON)": ["open", "high", "low", "close", "Fear & Greed Index"],
}
categorical_features = {
    "Рынок (CSV)": ["coin", "fear_greed_label", "halving_epoch", "golden_cross", "is_halving_day"],
    "Свечи BTC (JSON)": ["Fear & Greed Classification"],
    "Боты (XML)": ["record_type", "bot_category", "chain"],
    "Новости (CSV)": ["source"],
}
date_columns = {"Рынок (CSV)": "date", "Свечи BTC (JSON)": "timestamp", "Новости (CSV)": "createdAt"}

for name, frame in datasets.items():
    if frame is not None:
        print(f"\n{name}")
        print("Выбраны числовые:", [c for c in numeric_features.get(name, []) if c in frame])
        print("Выбраны категориальные:", [c for c in categorical_features.get(name, []) if c in frame])



Рынок (CSV)
Выбраны числовые: ['close', 'daily_return_pct', 'rolling_7d_vol', 'rolling_30d_vol', 'drawdown_from_ath_pct', 'volume', 'fear_greed_value', 'btc_dominance_pct', 'total_market_cap_usd', 'dxy_close', 'us10y_yield', 'days_since_halving']
Выбраны категориальные: ['coin', 'fear_greed_label', 'halving_epoch', 'golden_cross', 'is_halving_day']

Боты (XML)
Выбраны числовые: []
Выбраны категориальные: ['record_type', 'bot_category', 'chain']

Свечи BTC (JSON)
Выбраны числовые: ['open', 'high', 'low', 'close', 'Fear & Greed Index']
Выбраны категориальные: ['Fear & Greed Classification']

Новости (CSV)
Выбраны числовые: []
Выбраны категориальные: ['source']


## 5. Проверка типов и исследование пропусков

Числовые значения иногда записаны строками. Выбранные числовые поля приведём
к числовому типу, даты — к временному. Нераспознанные значения станут пропусками;
их количество отдельно покажет таблица преобразований.
Время приводится к UTC; для дат без часового пояса это рабочее допущение,
которое нужно подтвердить по описанию датасета.

**Исходные файлы не изменяются.** Преобразования выполняются в загруженных таблицах.


In [8]:
conversion_rows = []
for name, frame in datasets.items():
    if frame is None:
        continue
    columns = [c for c in numeric_features.get(name, []) if c in frame]
    date_col = date_columns.get(name)
    if date_col in frame:
        columns.append(date_col)
    for col in columns:
        missing_before = frame[col].isna().sum()
        if col == date_col:
            frame[col] = pd.to_datetime(frame[col], errors="coerce", format="mixed", utc=True)
        else:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
        conversion_rows.append({"Датасет": name, "Поле": col,
                                "Новых пропусков после преобразования": frame[col].isna().sum() - missing_before})

if conversion_rows:
    display(pd.DataFrame(conversion_rows))
    for name, col in date_columns.items():
        frame = datasets[name]
        if frame is not None and col in frame:
            print(f"{name}: {frame[col].min()} → {frame[col].max()}")
else:
    print("Проверка типов ожидает исходные файлы.")


,Датасет,Поле,Новых пропусков после преобразования
0,Рынок (CSV),close,0
1,Рынок (CSV),daily_return_pct,0
2,Рынок (CSV),rolling_7d_vol,0
3,Рынок (CSV),rolling_30d_vol,0
4,Рынок (CSV),drawdown_from_ath_pct,0
5,Рынок (CSV),volume,0
6,Рынок (CSV),fear_greed_value,0
7,Рынок (CSV),btc_dominance_pct,0
8,Рынок (CSV),total_market_cap_usd,0
9,Рынок (CSV),dxy_close,0


Рынок (CSV): 2017-01-01 00:00:00+00:00 → 2026-09-26 00:00:00+00:00
Свечи BTC (JSON): 2020-03-25 08:00:00+00:00 → 2026-09-30 04:00:00+00:00
Новости (CSV): 2026-07-01 17:51:50+00:00 → 2026-08-15 17:26:43+00:00


### 5.1. Количество и доля пропусков по столбцам

Для каждого поля рассчитывается:

$$\text{Доля пропусков, %} = \frac{\text{Число пропущенных значений}}{\text{Число строк}}\cdot 100.$$

Также показываются тип хранения и число уникальных заполненных значений.
Это помогает заметить полностью пустые поля и признаки, имеющие только одно значение.
Количество строк с хотя бы одним пропуском характеризует заполненность всей таблицы,
но само по себе не означает, что эти строки нужно удалять.


In [9]:
missing_reports = {}
for name, frame in datasets.items():
    if frame is None:
        continue
    report = pd.DataFrame({
        "Тип хранения": frame.dtypes.astype(str),
        "Пропусков": frame.isna().sum(),
        "Пропусков, %": (frame.isna().mean() * 100).round(2),
        "Уникальных без пропусков": frame.astype("string").nunique(),
    }).sort_values("Пропусков, %", ascending=False)
    missing_reports[name] = report
    print(f"\n{name}: {len(frame)} строк")
    display(report)
    print("Строк с хотя бы одним пропуском:", frame.isna().any(axis=1).sum())
    print("Полностью пустых столбцов:", list(frame.columns[frame.isna().all()]))

if not missing_reports:
    print("Для расчёта пропусков добавьте исходные CSV, JSON и XML.")



Рынок (CSV): 30120 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
fear_greed_label,object,2342,7.78,5
fear_greed_zone,object,2342,7.78,5
fear_greed_value,float64,2342,7.78,90
rolling_30d_vol,float64,20,0.07,30033
rolling_7d_vol,float64,20,0.07,30035
daily_return_pct,float64,10,0.03,30057
log_return,float64,10,0.03,26895
high,float64,0,0.00,29838
volume,int64,0,0.00,30118
close,float64,0,0.00,29856


Строк с хотя бы одним пропуском: 2348
Полностью пустых столбцов: []

Боты (XML): 220 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
bot_url,object,202,91.82,18
provider,object,176,80.00,6
model,object,176,80.00,6
tutorial_url,object,152,69.09,33
chain,object,129,58.64,36
bot_category,object,82,37.27,33
section,object,69,31.36,62
question,object,44,20.00,176
reliability,object,44,20.00,3
record_type,object,0,0.00,6


Строк с хотя бы одним пропуском: 220
Полностью пустых столбцов: []

Свечи BTC (JSON): 14280 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
Fear & Greed Classification,object,6,0.04,5
Fear & Greed Index,float64,6,0.04,88
timestamp,"datetime64[ns, UTC]",0,0.00,14280
close,float64,0,0.00,13786
open,float64,0,0.00,13791
low,float64,0,0.00,13478
high,float64,0,0.00,13509


Строк с хотя бы одним пропуском: 6
Полностью пустых столбцов: []

Новости (CSV): 413 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
thumbnail,object,1,0.24,403
url,object,0,0.00,413
title,object,0,0.00,412
description,object,0,0.00,412
createdAt,"datetime64[ns, UTC]",0,0.00,412
source,object,0,0.00,10


Строк с хотя бы одним пропуском: 1
Полностью пустых столбцов: []


### 5.2. Простое уточнение: пропуски внутри групп

В рыночном CSV полезно проверить, одинаково ли заполнены данные разных монет.
В XML поля могут относиться только к определённым типам записей: например,
отсутствие `bot_category` у вопроса и ответа не обязательно является ошибкой.
Поэтому дополнительно посмотрим долю пропусков внутри этих групп.


In [10]:
if df_market is not None and "coin" in df_market:
    cols = [c for c in ["close", "volume", "fear_greed_value", "rolling_30d_vol"] if c in df_market]
    if cols:
        print("Рыночный CSV: пропуски по монетам, %")
        display(df_market.groupby("coin", dropna=False)[cols].agg(lambda s: s.isna().mean() * 100).round(2))

if df_bots is not None and "record_type" in df_bots:
    cols = [c for c in ["bot_category", "chain", "question", "answer"] if c in df_bots]
    if cols:
        print("XML: пропуски по типам записей, %")
        display(df_bots.groupby("record_type", dropna=False)[cols].agg(lambda s: s.isna().mean() * 100).round(2))


Рыночный CSV: пропуски по монетам, %


,close,volume,fear_greed_value,rolling_30d_vol
coin,,,,
ADA,0.0,0.0,7.55,0.06
AVAX,0.0,0.0,0.05,0.09
BNB,0.0,0.0,7.55,0.06
BTC,0.0,0.0,15.66,0.06
DOGE,0.0,0.0,7.55,0.06
DOT,0.0,0.0,0.04,0.09
ETH,0.0,0.0,7.55,0.06
LTC,0.0,0.0,15.66,0.06
SOL,0.0,0.0,0.04,0.08


XML: пропуски по типам записей, %


,bot_category,chain,question,answer
record_type,,,,
bot_profile,0.0,11.11,0.0,0.0
chain_profile,0.0,0.00,0.0,0.0
llm_consensus,0.0,100.00,100.0,0.0
qa_pair,0.0,0.00,0.0,0.0
real_test_data,100.0,100.00,0.0,0.0
verified_fact,100.0,100.00,0.0,0.0


### 5.3. Как интерпретировать пропуски на начальном этапе

| Где обнаружен пропуск | Возможная причина — требует проверки | Первоначальное решение |
|---|---|---|
| Первая доходность | Нет предыдущей цены | Сохранить пропуск |
| Начало `rolling_7d_vol`, `rolling_30d_vol` | Недостаточно дней для окна | Сохранить пропуск; проверить правило расчёта |
| `fear_greed_value`, `dxy_close`, `us10y_yield` | Другой период покрытия или календарь источника | Уточнить доступность данных по датам |
| `close`, `date`, `coin` | Неполная запись или ошибка чтения | Проверить отдельно: это важные поля наблюдения |
| `description` новости | Источник не передал описание | Использовать доступный заголовок при дальнейшем анализе |
| Поле XML только у некоторых типов записей | Поле неприменимо к другим типам | Оценивать полноту внутри `record_type` |

Сейчас достаточно зафиксировать пропуски и возможные причины.
**Автоматическое заполнение нулями, средними значениями или массовое удаление строк
на этом этапе не выполняется.** Сначала нужно понять смысл каждого поля.


## 6. Минимальная описательная статистика выбранных признаков

Для числовых полей посмотрим число заполненных значений, среднее, медиану,
минимум и максимум. Эти значения помогают заметить необычный масштаб или возможные ошибки.
Например, у цены нужно проверить нулевые и отрицательные значения, а у индекса
страха и жадности — значения вне шкалы 0–100.

Для категориальных полей посчитаем частоты значений. Пропуск будет показан
отдельной строкой. Максимум десять наиболее частых категорий выводятся для краткости.


In [11]:
for name, frame in datasets.items():
    if frame is None or frame.empty:
        continue
    cols = [c for c in numeric_features.get(name, []) if c in frame]
    if cols:
        print(f"\n{name}: числовые характеристики")
        stats = frame[cols].describe().T[["count", "mean", "50%", "min", "max"]]
        stats.columns = ["Заполнено", "Среднее", "Медиана", "Минимум", "Максимум"]
        display(stats)
    for col in categorical_features.get(name, []):
        if col in frame:
            print(f"{name}: {col}")
            display(frame[col].value_counts(dropna=False).head(10).rename("Количество").to_frame())



Рынок (CSV): числовые характеристики


,Заполнено,Среднее,Медиана,Минимум,Максимум
close,30120.0,4.556112e+03,2.738360e+01,1.038000e-03,1.247525e+05
daily_return_pct,30110.0,2.412812e-01,9.127000e-03,-4.234722e+01,3.555466e+02
rolling_7d_vol,30100.0,4.316715e+00,3.522285e+00,1.304300e-01,1.372630e+02
rolling_30d_vol,30100.0,4.709001e+00,4.034044e+00,1.304300e-01,6.753813e+01
drawdown_from_ath_pct,30120.0,-6.227310e+01,-7.061443e+01,-9.860659e+01,0.000000e+00
volume,30120.0,6.030088e+09,1.139117e+09,3.957000e+03,3.509679e+11
fear_greed_value,27778.0,4.667370e+01,4.700000e+01,5.000000e+00,9.500000e+01
btc_dominance_pct,30120.0,5.202889e+01,5.416000e+01,3.500000e+01,8.700000e+01
total_market_cap_usd,30120.0,2.888259e+12,2.888259e+12,2.888259e+12,2.888259e+12
dxy_close,30120.0,9.917168e+01,9.881000e+01,8.907000e+01,1.133100e+02


Рынок (CSV): coin


,Количество
coin,
LTC,3556
BTC,3556
XRP,3244
ADA,3244
DOGE,3244
BNB,3244
ETH,3244
SOL,2361
DOT,2229


Рынок (CSV): fear_greed_label


...

,Заполнено,Среднее,Медиана,Минимум,Максимум
open,14280.0,51488.636947,46336.25,5873.0,125496.0
high,14280.0,51875.182892,46761.35,5950.0,126080.0
low,14280.0,51085.205014,45710.00,5841.5,124863.0
close,14280.0,51494.136597,46343.85,5873.0,125375.0
Fear & Greed Index,14274.0,47.950960,49.00,5.0,95.0


Свечи BTC (JSON): Fear & Greed Classification


,Количество
Fear & Greed Classification,
Greed,4118
Fear,3444
Extreme Fear,3154
Neutral,1992
Extreme Greed,1566
None,6


Новости (CSV): source


,Количество
source,
cryptodaily,200
bsc,40
cryptopotato,36
decrypt,35
cointelegraph,30
coindesk,25
theguardian,20
bitcoinmagazine,10
coinjournal,9
